# Isolation Forest

Mỗi KPI được huấn luyện & tune không dùng chung tham số, so sánh 2 route:
- **Route (a)** shingle: tune `shingle_size × max_samples` trên val của từng KPI.
- **Route (b)** feature: `build_features` + `select_features` (từ `FE_FS/Code/features.py`) rồi fit IF.

Mọi KPI dùng chung `preprocess.py` + `eval_protocol.py` → ngưỡng tune trên **val**, đo trên **test**
Kết quả: bảng params + ngưỡng + metrics per-KPI, và macro AP để so 2 route.

In [1]:
import sys, time, warnings
from pathlib import Path
def find_root(marker='Data/train.csv'):
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f'Không thấy repo root từ {Path.cwd()}')
ROOT = find_root()
sys.path.append(str(ROOT / 'Modeling' / 'Code'))
sys.path.append(str(ROOT / 'FE_FS' / 'Code'))
warnings.filterwarnings("ignore")

In [2]:
import numpy as np, pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from eval_protocol import time_split_per_kpi, evaluate_protocol
from preprocess import preprocess_all
from features import build_features, select_features

In [3]:
df = pd.read_csv(ROOT / 'Data' / 'train.csv')
df.columns = ["timestamp", "value", "label", "kpi"]
SHINGLE_GRID = (8, 16, 32, 48, 64); MS_GRID = (128, 256, 512)

In [4]:
def run_kpi(gk):
    step = int(pd.Series(np.diff(np.sort(gk.timestamp.values))).mode().iloc[0])
    gk = time_split_per_kpi(gk, train_frac=0.6, val_frac=0.2)
    p = preprocess_all(gk, max_gap_points=5, norm_method="robust").sort_values("timestamp").reset_index(drop=True)
    y = p.label.values.astype(int); spl = p.split.values

    if y[spl == "val"].sum() == 0 or y[spl == "test"].sum() == 0:
        return None
    out = dict(kpi=gk.kpi.iloc[0][:8], n_anom_test=int(y[spl == "test"].sum()))

    # ---- ROUTE A: shingle ----
    v = p.value_norm.values.astype(float)
    def shingle_score(S, MS, seed):
        W = sliding_window_view(v, S) 
        cur = np.arange(S-1, len(v)) 
        ok = ~np.isnan(W).any(1)
        Xok, spok = W[ok], spl[cur][ok]
        sc = -IsolationForest(n_estimators=100, max_samples=MS, contamination="auto",
                              random_state=seed).fit(Xok[spok == "train"]).decision_function(Xok)
        
        full = np.full(len(v), np.nan) 
        full[cur[ok]] = sc
        return full
    
    bestA = None                                                 # tune trên VAL
    for S in SHINGLE_GRID:
        for MS in MS_GRID:
            s = shingle_score(S, MS, 42)
            vm = (spl == "val") & ~np.isnan(s)
            ap = average_precision_score(y[vm], s[vm]) if y[vm].sum() else -1
            if bestA is None or ap > bestA[0]: bestA = (ap, S, MS)
    _, S, MS = bestA
    s = shingle_score(S, MS, 0)
    vm = (spl == "val") & ~np.isnan(s)
    tm = (spl == "test") & ~np.isnan(s)
    rA = evaluate_protocol(y[vm], s[vm], y[tm], s[tm], step_s=step)

    out.update(a_shingle=S, a_max_samples=MS, a_AP=round(rA["AP_pw"], 3),
               a_PW_F1=round(rA["PW"]["fbeta"], 3), a_PA_F1=round(rA["PA"]["fbeta"], 3),
               a_thr_pw=round(rA["PW"]["threshold"], 4))

    # ---- ROUTE B: feature ----
    F = build_features(p); final = select_features(F, p)
    X = F[final].values; valid = ~np.isnan(X).any(1); tr = spl == "train"

    def feat_score(MS, seed):
        return -IsolationForest(n_estimators=100, max_samples=MS, contamination="auto",
                                random_state=seed).fit(X[tr & valid]).decision_function(X)

    bestB = None
    for MS in MS_GRID:
        s = feat_score(MS, 42); vm = (spl == "val") & valid
        ap = average_precision_score(y[vm], s[vm]) if y[vm].sum() else -1
        if bestB is None or ap > bestB[0]: bestB = (ap, MS)
    _, MSb = bestB
    sc = feat_score(MSb, 0)
    vm = (spl == "val") & valid; tm = (spl == "test") & valid
    rB = evaluate_protocol(y[vm], sc[vm], y[tm], sc[tm], step_s=step)
    out.update(b_n_feat=len(final), b_max_samples=MSb, b_AP=round(rB["AP_pw"], 3),
               b_PW_F1=round(rB["PW"]["fbeta"], 3), b_PA_F1=round(rB["PA"]["fbeta"], 3),
               b_thr_pw=round(rB["PW"]["threshold"], 4))
    return out

In [5]:
rows, skipped = [], []
t0 = time.time()
for kpi, gk in df.groupby("kpi"):
    r = run_kpi(gk.copy())
    if r is None:
        skipped.append(kpi[:8])
    else:
        rows.append(r); print(f"done {r['kpi']} | a_AP {r['a_AP']} b_AP {r['b_AP']}")
print(f"Skipped ({len(skipped)}):", skipped)
print(f"(time {time.time()-t0:.0f}s)")

done 02e99bd4 | a_AP 0.796 b_AP 0.898
done 07927a9a | a_AP 0.038 b_AP 0.046
done 09513ae3 | a_AP 0.009 b_AP 0.013
done 18fbb1d5 | a_AP 0.116 b_AP 0.636
done 1c35dbf5 | a_AP 0.976 b_AP 0.979
done 40e25005 | a_AP 0.02 b_AP 0.207
done 71595dd7 | a_AP 0.01 b_AP 0.202
done 7c189dd3 | a_AP 0.068 b_AP 0.617
done 88cf3a77 | a_AP 0.017 b_AP 0.244
done 8bef9af9 | a_AP 0.01 b_AP 0.555
done 8c892e55 | a_AP 0.861 b_AP 0.382
done 9ee58794 | a_AP 0.072 b_AP 0.906
done a40b1df8 | a_AP 0.058 b_AP 0.628
done affb01ca | a_AP 0.006 b_AP 0.538
done c58bfcba | a_AP 0.003 b_AP 0.004
done cff6d3c0 | a_AP 0.024 b_AP 0.17
done da403e4e | a_AP 0.582 b_AP 0.829
done e0770391 | a_AP 0.874 b_AP 0.39
Skipped (8): ['046ec29d', '54e8a140', '769894ba', '76f4550c', '8a20c229', '9bd90500', 'a5bf5d65', 'b3b2e6d1']
(time 526s)


In [6]:
res = pd.DataFrame(rows).sort_values("b_AP", ascending=False)
print(f"MACRO AP    | route a = {res['a_AP'].mean():.3f} | route b = {res['b_AP'].mean():.3f}")
print(f"MACRO PW_F1 | route a = {res['a_PW_F1'].mean():.3f} | route b = {res['b_PW_F1'].mean():.3f}")
win_b = (res['b_AP'] > res['a_AP']).sum()
print(f"Route (theo AP): route b {win_b}/{len(res)} | route a {len(res)-win_b}/{len(res)}")
res

MACRO AP    | route a = 0.252 | route b = 0.458
MACRO PW_F1 | route a = 0.155 | route b = 0.344
Route (theo AP): route b 16/18 | route a 2/18


,kpi,n_anom_test,a_shingle,a_max_samples,a_AP,a_PW_F1,a_PA_F1,a_thr_pw,b_n_feat,b_max_samples,b_AP,b_PW_F1,b_PA_F1,b_thr_pw
4,1c35dbf5,2709,64,512,0.976,0.687,0.000,0.0426,25,256,0.979,0.768,0.998,0.0733
11,9ee58794,761,8,256,0.072,0.120,0.195,-0.0091,14,512,0.906,0.701,0.643,0.1117
0,02e99bd4,1382,48,512,0.796,0.674,0.815,-0.0003,19,256,0.898,0.754,0.991,0.0327
16,da403e4e,238,8,512,0.582,0.618,0.853,0.0552,21,512,0.829,0.339,0.932,-0.0240
3,18fbb1d5,47,32,512,0.116,0.233,0.415,0.0036,23,512,0.636,0.255,0.784,0.0174
12,a40b1df8,86,8,512,0.058,0.100,0.175,0.0413,15,128,0.628,0.580,0.455,0.1457
7,7c189dd3,63,8,512,0.068,0.143,0.252,0.0890,16,128,0.617,0.542,0.403,0.1526
9,8bef9af9,73,8,512,0.010,0.019,0.020,0.1021,15,512,0.555,0.520,0.293,0.2083
13,affb01ca,76,8,512,0.006,0.015,0.020,0.1200,15,512,0.538,0.512,0.347,0.2289
17,e0770391,2706,16,256,0.874,0.000,0.000,0.2341,15,512,0.390,0.086,0.152,0.0821


In [8]:
res.to_json(ROOT / 'Modeling' / 'ML' / 'IsolationForest' / 'if_per_kpi_config.json',
            orient="records", indent=1)